In [ ]:
import sys
sys.path.append("..")

import jax, os, corner
import jax.numpy as jnp
from jax import grad, config
import matplotlib.pyplot as plt
import numpy as np
config.update("jax_enable_x64", True)
config.update("jax_debug_nans", True)
from functools import partial
from models.priors import Mc_eta_uniform_masses_draw
from models.gw150914 import gwfast_LVGW150914

from src.pampel import ula_sampler_full_jax_jit
from src.helper import rejection_sampling

print(jax.devices())

In [ ]:
# Initialize model
model = gwfast_LVGW150914(wf_model='IMRPhenomD', nbins=100, verbose=True)

In [ ]:
# Setup and run sampler
n_iter = 100
n_particles = 200
eps = 1e-6

# Birth death
stride = n_iter + 1 
rate = 1e-6 
bandwidth = jnp.ones(model.DoF) * 0.01
bandwidth = bandwidth.at[jnp.array([4, 6, 8])].set(jnp.ones(3) * 0.01)

# Initial draw from prior
X0 = model._newDrawFromPrior(n_particles)
X0 = X0.at[:, jnp.array([0, 1])].set(Mc_eta_uniform_masses_draw(n_particles, model.lower_bound[0:2], model.upper_bound[0:2]))

key = jax.random.PRNGKey(0)

sam = ula_sampler_full_jax_jit(key, model.minusLogLikelihood, model.gradient_minusLogLikelihood, n_iter, eps, X0, model.lower_bound, model.upper_bound, stride, rate, bandwidth, model.bounded_coordinates, model.periodic_coordinates)

In [ ]:
reshaped_matrix = np.array(sam.reshape((sam.shape[0] * sam.shape[1], 11)))
fig = corner.corner(reshaped_matrix[-50000:], hist_kwargs={'density':True}, labels=model.gwfast_param_order, truths=model.true_params)